In [ ]:
%%capture
!pip install xclim==0.36.0 
import xarray as xr
import intake
import matplotlib.pyplot as plt

from xclim.core.calendar import convert_calendar
from xclim.core.units import convert_units_to
from xclim.sdba.adjustment import QuantileDeltaMapping

%config InlineBackend.figure_format = 'svg' # Make plots look better in the notebook environment

Utility function

In [ ]:
import pyproj
import rioxarray as rio

def get_closest_gridcell(data, lat, lon):
    # Make Transformer object
    lat_lon_to_model_projection = pyproj.Transformer.from_crs(
        crs_from="epsg:4326",  # Lat/lon
        crs_to=data.rio.crs,  # Model projection
        always_xy=True,
    )

    # Convert coordinates to x,y
    x, y = lat_lon_to_model_projection.transform(lon, lat)

    # Get closest gridcell
    closest_gridcell = data.sel(x=x, y=y, method="nearest")

    # Output information
    print(
        "Input coordinates: (%.2f, %.2f)" % (lat, lon)
        + "\nNearest grid cell coordinates: (%.2f, %.2f)"
        % (closest_gridcell.lat.values.item(), closest_gridcell.lon.values.item())
    )
    return closest_gridcell

Open the HadISD dataset for Sacramento Executive Airport (KSAC) and process

In [ ]:
obs_ds = xr.open_dataset('station-data/hadisd.3.3.0.202202p_19310101-20220301_724830-23232_KSAC_temperatures.nc')
obs_ds = obs_ds.sel(time=slice('1981','2010'))
obs_ds = obs_ds.temperatures
obs_ds.attrs["units"] = "degF"
obs_ds = convert_calendar(obs_ds, "noleap")
obs_ds = obs_ds.compute()

Extract the coordinates of the station

In [ ]:
lat0 = obs_ds.latitude.values
lon0 = obs_ds.longitude.values

Get the WRF data from the CAE catalog

In [ ]:
col = intake.open_esm_datastore('https://cadcat.s3.amazonaws.com/cae-collection.json')
cat = col.search(
    table_id = "1hr",
    grid_label = "d03",
    variable_id = "t2",
    experiment_id = ["historical","ssp370"],
    source_id = "CESM2"
)
dsets = cat.to_dataset_dict(zarr_kwargs={'consolidated': True}, 
                            storage_options={'anon': True},
                           )
# Subsets the historical scenario
hist_dsets = {key: val for key,val in dsets.items()
             if "historical" in key}
# Subsets the future scenario
ssp_dsets = {key: val for key,val in dsets.items()
               if "ssp370" in key}

hist_ds = list(hist_dsets.values())[0]
ssp_ds = list(ssp_dsets.values())[0]

Extract the WRF grid cell closest to the station and process the data

In [ ]:
# define 30-year windows
hist_ds = hist_ds.sel(time=slice('1981','2010')).t2.squeeze()
ssp_ds = ssp_ds.sel(time=slice('2031','2060')).t2.squeeze()

# convert units
hist_ds = convert_units_to(hist_ds, "degF")
ssp_ds = convert_units_to(ssp_ds, "degF")

# convert calendar
hist_ds = convert_calendar(hist_ds, "noleap")
ssp_ds = convert_calendar(ssp_ds, "noleap")

# extract closest grid cell
hist_ds = get_closest_gridcell(hist_ds,
                               lat0,lon0)
ssp_ds = get_closest_gridcell(ssp_ds,
                               lat0,lon0)

# need to unchunk for bias correction
hist_ds = hist_ds.chunk(dict(
    time=-1)).compute()
ssp_ds = ssp_ds.chunk(dict(
    time=-1)).compute()

Inspect annual mean daily temperatures for the observations and raw WRF data

In [ ]:
fig, ax = plt.subplots()
hist_ds.groupby('time.dayofyear').mean().plot(ax=ax,label='Raw historical model',c='tab:blue')
ssp_ds.groupby('time.dayofyear').mean().plot(ax=ax,label='Raw projected model',c='tab:orange')
obs_ds.groupby('time.dayofyear').mean().plot(ax=ax,label='Observations',c='k',lw=1)
ax.set_title('Record-mean daily-mean temperature at 2m')
ax.set_ylabel("2m air temperature (F)")
ax.set_xlabel("Day of year")
ax.legend()
plt.show()

Create the training set (i.e., find the adjustment factors). This will speedup the adjustment process later on.

In [ ]:
QDM = QuantileDeltaMapping.train(
    obs_ds, hist_ds, nquantiles=20, 
    group = 'time.dayofyear', kind="+"
)

print("Lowest quantile which will be ajusted is " 
      + str(QDM.ds.quantiles.values.min()))
print("Highest quantile which will be ajusted is " 
      + str(QDM.ds.quantiles.values.max()))

Inspect the raw historical WRF quantiles and the adjustment factor for each quantile

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(10,4), sharey=True)
ax = ax.ravel()

hist_q_plot = QDM.ds['hist_q'].plot(ax=ax[0])
ax[0].set_title('Raw WRF historical quantiles')
ax[0].set_ylabel('Day of year')
af_plot = QDM.ds['af'].plot( ax=ax[1])
ax[1].set_title('Adjustment factor by quantile')
ax[1].set_ylabel('')

plt.show()

Perform the data adjustment for the historical and projected data

In [ ]:
hist_adj = QDM.adjust(hist_ds).compute()
ssp_adj = QDM.adjust(ssp_ds).compute()


Check out some results!

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,4),sharey=True)
ax = ax.ravel()
hist_ds.groupby('time.dayofyear').mean(
).plot(ax=ax[0],label='Raw',alpha=1)
hist_adj.groupby('time.dayofyear').mean(
).plot(ax=ax[0],label='Adjusted',
       c='goldenrod',alpha=1,lw=2)
obs_ds.groupby('time.dayofyear').mean(
).plot(ax=ax[0],c='k',ls='-',lw=.25,
      label='Observations')
ax[0].set_title("Historical")
ax[0].set_ylabel("2m air temperature (F)")
ax[0].set_xlabel("Day of year")
ax[0].legend()

ssp_ds.groupby('time.dayofyear').mean(
).plot(ax=ax[1],label='Raw',alpha=1)
ssp_adj.groupby('time.dayofyear').mean(
).plot(ax=ax[1],label='Adjusted',
       c='goldenrod',alpha=1,lw=1.5)
ax[1].set_title("Projected")
ax[1].set_xlabel("Day of year")
ax[1].set_ylabel("")
ax[1].legend()

plt.suptitle('Record-mean daily-mean temperature at 2m')

plt.show()

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,4),sharey=True)
ax = ax.ravel()
hist_ds.groupby('time.year').mean(
).plot(ax=ax[0],label='Raw',alpha=1)
hist_adj.groupby('time.year').mean(
).plot(ax=ax[0],label='Adjusted',
       c='goldenrod',alpha=1,lw=1.5)
obs_ds.groupby('time.year').mean(
).plot(ax=ax[0],c='k',ls='-',lw=.5,
      label='Observations')
ax[0].set_title("Historical")
ax[0].set_ylabel("2m air temperature (F)")
ax[0].set_xlabel("Year")
ax[0].legend()

ssp_ds.groupby('time.year').mean(
).plot(ax=ax[1],label='Raw',alpha=1)
ssp_adj.groupby('time.year').mean(
).plot(ax=ax[1],label='Adjusted',
       c='goldenrod',alpha=1,lw=1.5)
ax[1].set_title("Projected")
ax[1].set_xlabel("Year")
ax[1].set_ylabel("")
ax[1].legend()

plt.suptitle('Annual mean temperature at 2m')
plt.show()

Let's look at some extreme values (max and min temperatures) and how they are adjusted

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,4),sharey=True)
ax = ax.ravel()
hist_ds.isel(time=slice(hist_ds.argmax().values-72,
                        hist_ds.argmax().values+72)).plot(
color='tab:blue',label='Raw',ax=ax[0])
hist_adj.isel(time=slice(hist_ds.argmax().values-72,
                         hist_ds.argmax().values+72)).plot(
color='goldenrod',label='Adjusted',ax=ax[0])
ax[0].set_title("Historical")
ax[0].set_ylabel("2m air temperature (F)")
ax[0].set_xlabel("Time")
ax[0].legend()

ssp_ds.isel(time=slice(ssp_ds.argmax().values-72,
                        ssp_ds.argmax().values+72)).plot(
color='tab:blue',label='Raw',ax=ax[1])
ssp_adj.isel(time=slice(ssp_adj.argmax().values-72,
                         ssp_adj.argmax().values+72)).plot(
color='goldenrod',label='Adjusted',ax=ax[1])
ax[1].set_title("Projected")
ax[1].set_xlabel("Time")
ax[1].set_ylabel("")
ax[1].legend()

plt.suptitle('Maximum hourly temperature +/- 72 hours')

plt.show()

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,4),sharey=True)
ax = ax.ravel()
hist_ds.isel(time=slice(hist_ds.argmin().values-72,
                        hist_ds.argmin().values+72)).plot(
color='tab:blue',label='Raw',ax=ax[0])
hist_adj.isel(time=slice(hist_ds.argmin().values-72,
                         hist_ds.argmin().values+72)).plot(
color='goldenrod',label='Adjusted',ax=ax[0])
ax[0].set_title("Historical")
ax[0].set_ylabel("2m air temperature (F)")
ax[0].set_xlabel("Time")
ax[0].legend()

ssp_ds.isel(time=slice(ssp_ds.argmin().values-72,
                        ssp_ds.argmin().values+72)).plot(
color='tab:blue',label='Raw',ax=ax[1])
ssp_adj.isel(time=slice(ssp_adj.argmin().values-72,
                         ssp_adj.argmin().values+72)).plot(
color='goldenrod',label='Adjusted',ax=ax[1])
ax[1].set_title("Projected")
ax[1].set_xlabel("Time")
ax[1].set_ylabel("")
ax[1].legend()

plt.suptitle('Minimum hourly temperature +/- 72 hours')

plt.show()